# Chunk Strategies for RAG

Chunking configuration influences retrieval quality as much as the choice of embedding model. Get chunking wrong and no amount of reranking saves you.

## Problem Definition

...

## Basic Concept

### Fixed chunking

Split every N characters or tokens.

Simplest, breaks mid-sentence, good comppression, bad coherence

### Recursive

Try splitting on `\n\n` first, then `\n`, then `.`, then space. Falls back cleanly.

### Semantic

Embed each sentence. Compute the cosine similarity between adjacent sentences. Split where similarity drops below a threshold. 

Perserve topic coherence, Slower. Sometimes produces tiny 40-token fragments that hurt retrieval.

### Sentence

Split on sentence boundaries. One sentence per chunk or a window of N sentences. Matches semantic chunking up to ~5k tokens at a fraction of the cost

### Parent-document

Store small child chunks for retrieval and the larger parent chunk for context.

Retrieve by child, return parent. Degrades gracefully: bad child chunks still return reasonable parents.

### Late chunking

Embed the whole document at the token level first, then pool token embeddings into chunk embeddings. Preserves cross-chunk context. Works with long-context embedders. Higher compute.

### Contextual retrieval

Perpend each chunk with an LLM-generated summary of its position in the document.

("This chunk is section 3.2 of the termination clauses...").. Expensive to index

## Chunk size

Match the chunk size to the query type:
||Query type|Chunk size|
|---|---|---|
||Factoid("What is the CEO's name?")|256~512 tokens|
||Analytical/multi-hop|512~1024 tokens|
||Whole-section comprehension|1024~2048 tokens|

# Build you Own

## Fixed and recursive chunking

In [2]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../../00_Common").resolve()))
from user_tools import SectionPrinter

def chunk_fixed(text, chunk_size, overlap=0):
    step = chunk_size - overlap
    return [text[i:i+chunk_size] for i in range(0, len(text), step)]

def chunk_recursive(text, size=512, seps=("\n\n", "\n", ". ", " ")):
    if len(text) <= size:
        return [text]

    for sep in seps:
        if sep not in text:
            continue
        parts = text.split(sep)
        chunks = []
        buf = ""
        for p in parts:
            if len(p) > size:
                if buf:
                    chunks.append(buf)
                    buf = ""
                chunks.extend(chunk_recursive(p, size=size, seps=seps[1:] or (" ",)))
                continue
            candidate = buf + sep + p if buf else p
            if len(candidate) <= size:
                buf = candidate
            else:
                if buf:
                    chunks.append(buf)
                buf = p
        if buf:
            chunks.append(buf)
        return [c for c in chunks if c.strip()]
    return chunk_fixed(text, size)

In [7]:
SAMPLE_DOCUMENT = """
RAG Chunking Policy

Section 1: Purpose
Retrieval-augmented generation depends on how documents are split before indexing.
If chunks are too large, embeddings become vague. If chunks are too small, context is lost.

Section 2: Fixed Chunking
Fixed chunking splits text every N characters. It is fast and predictable.
However, it may cut a sentence in half, which hurts readability and retrieval quality.

Section 3: Recursive Chunking
Recursive chunking tries paragraph breaks first, then line breaks, then sentence boundaries.
This usually keeps related sentences together and produces more coherent chunks for search.

Section 4: Practical Guidance
For factoid questions, prefer smaller chunks around 256 to 512 tokens.
For analytical questions, use larger chunks with some overlap between adjacent segments.
Always inspect a few chunks manually before building a production index.
""".strip()

def show_chunks(title, chunks, preview=80):
    print(f"{title}: {len(chunks)} chunks")
    for i, chunk in enumerate(chunks, 1):
        snippet = chunk.replace("\n", " ")
        if len(snippet) > preview:
            snippet = snippet[:preview] + "..."
        print(f"  [{i}] ({len(chunk)} chars) {snippet}")

with SectionPrinter("Fixed chunking"):
    fixed_chunks = chunk_fixed(SAMPLE_DOCUMENT, chunk_size=120, overlap=20)
    show_chunks("fixed", fixed_chunks)

with SectionPrinter("Recursive chunking"):
    recursive_chunks = chunk_recursive(SAMPLE_DOCUMENT, size=120)
    show_chunks("recursive", recursive_chunks)

=======================Fixed chunking=======================
fixed: 9 chunks
  [1] (120 chars) RAG Chunking Policy  Section 1: Purpose Retrieval-augmented generation depends o...
  [2] (120 chars) split before indexing. If chunks are too large, embeddings become vague. If chun...
  [3] (120 chars) ntext is lost.  Section 2: Fixed Chunking Fixed chunking splits text every N cha...
  [4] (120 chars) and predictable. However, it may cut a sentence in half, which hurts readability...
  [5] (120 chars) ty.  Section 3: Recursive Chunking Recursive chunking tries paragraph breaks fir...
  [6] (120 chars) , then sentence boundaries. This usually keeps related sentences together and pr...
  [7] (120 chars)  chunks for search.  Section 4: Practical Guidance For factoid questions, prefer...
  [8] (120 chars) nd 256 to 512 tokens. For analytical questions, use larger chunks with some over...
  [9] (83 chars)  segments. Always inspect a few chunks manually before building a production ind...
======

## Semantic chunking

In [6]:
from encodings import normalize_encoding
import re
from sentence_transformers import SentenceTransformer


def split_sentences(text):
    parts = re.split(r"(?<=[.!?])\s+", text.strip())
    return [p.strip() for p in parts if p.strip()]

def chunk_semantic(text, encoder, threshold = 0.6, min_chars=200, max_chars=2048):
    sentences = split_sentences(text)
    if not sentences:
        return []
    embs = encoder.encode(sentences, normalize_embeddings=True)
    chunks = [[sentences[0]]]
    for i in range(1, len(sentences)):
        sim = float(embs[i] @ embs[i-1])
        current_len = sum(len(s) for s in chunks[-1])
        if sim < threshold and current_len >= min_chars:
            chunks.append([sentences[i]])
        else:
            chunks[-1].append(sentences[i])
    result = []
    for group in chunks:
        text_group = "".join(group)
        if len(text_group) > max_chars:
            result.extend(chunk_recursive(text_group, size=max_chars))
        else:
            result.append(text_group)
    return result

with SectionPrinter("Semantic chunking"):
    encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    chunks = chunk_semantic(SAMPLE_DOCUMENT, encoder, threshold=0.6, min_chars=200, max_chars=2048)
    show_chunks("semantic", chunks)


W0807 12:36:23.677000 2096 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
W0807 12:36:23.723000 2096 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0807 12:36:23.765000 2096 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


=====================Semantic chunking======================


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

semantic: 4 chunks
  [1] (212 chars) RAG Chunking Policy  Section 1: Purpose Retrieval-augmented generation depends o...
  [2] (307 chars) Section 2: Fixed Chunking Fixed chunking splits text every N characters.It is fa...
  [3] (279 chars) This usually keeps related sentences together and produces more coherent chunks ...
  [4] (72 chars) Always inspect a few chunks manually before building a production index.


## Parent document

In [9]:
import numpy as np

def chunk_parent_child(text, parent_size=2048, child_size=256):
    parents = chunk_recursive(text, size=parent_size)
    mapping = []
    for p_idx, parent in enumerate(parents):
        children = chunk_recursive(parent, size=child_size)
        for child in children:
            mapping.append({
                "child": child,
                "parent_idx": p_idx,
                "parent": parent
            })

    return mapping

def retrieve_parent(child_query, mapping, encoder, top_k=3):
    child_embs = encoder.encode([m["child"] for m in mapping], normalize_embeddings=True)
    q_emb = encoder.encode([child_query], normalize_embeddings=True)[0]
    scores = child_embs @ q_emb

    top = np.argsort(-scores)[:top_k]
    seen, parents = set(), []
    for i in top:
        if mapping[i]["parent_idx"] not in seen:
            parents.append(mapping[i]["parent"])
            seen.add(mapping[i]["parent_idx"])
    return parents

with SectionPrinter("Parent document"):
    mapping = chunk_parent_child(SAMPLE_DOCUMENT)
    print(retrieve_parent("What is the CEO's name?", mapping, encoder))


======================Parent document=======================
['RAG Chunking Policy\n\nSection 1: Purpose\nRetrieval-augmented generation depends on how documents are split before indexing.\nIf chunks are too large, embeddings become vague. If chunks are too small, context is lost.\n\nSection 2: Fixed Chunking\nFixed chunking splits text every N characters. It is fast and predictable.\nHowever, it may cut a sentence in half, which hurts readability and retrieval quality.\n\nSection 3: Recursive Chunking\nRecursive chunking tries paragraph breaks first, then line breaks, then sentence boundaries.\nThis usually keeps related sentences together and produces more coherent chunks for search.\n\nSection 4: Practical Guidance\nFor factoid questions, prefer smaller chunks around 256 to 512 tokens.\nFor analytical questions, use larger chunks with some overlap between adjacent segments.\nAlways inspect a few chunks manually before building a production index.']


## Contextual Retrieval

In [ ]:
def contextualize_chunks(document, chunks, llm):
    context_prompts = [
        f"""<document>{document}</document>
        Here is the chunk to situate: <chunk>{c}</chunk>
        Write 50-100 words placing this chunk in the document's context.
        """
        for c in chunks
    ]

    contexts = llm.batch(context_prompts)
    return [f"{ctx}\n\n{c}" for ctx, c in zip(contexts, chunks)]